In [9]:
import os
import sys
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

spark = SparkSession.builder \
    .appName("BankTransactionsRDD") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.pyspark.python", sys.executable) \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark is ready!")

Spark is ready!


In [22]:
from pathlib import Path

file_path = str(Path.cwd() / "bank.txt")
data = sc.textFile(file_path)

header = data.first()

transactions = (
    data.filter(lambda line: line.strip() and line.strip() != header.strip())
        .map(lambda line: [field.strip() for field in line.split(",")])
        .filter(lambda row: len(row) == 5)
        .map(lambda row: (
            row[0],        # Bank ID
            row[1],        # Account Number
            row[2],        # Transaction Date
            row[3],        # Transaction Type
            float(row[4])  # Transaction Amount
        ))
)

print("Total transactions:", transactions.count())
print("Sample records:", transactions.take(5))

Total transactions: 15
Sample records: [('B001', '1001', '05-01-2024', 'Credit', 5000.0), ('B001', '1002', '10-02-2024', 'Debit', 1500.0), ('B002', '1003', '15-03-2024', 'Credit', 7000.0), ('B001', '1001', '20-04-2024', 'Debit', 2000.0), ('B002', '1004', '12-05-2024', 'Credit', 3500.0)]


In [29]:
#1.Count unique number of customers
print("Unique customers:", transactions
      .map(lambda x: x[1]).distinct().count())

Unique customers: 7


In [30]:
#2.Count unique number of Bank IDs
print("Unique Bank IDs:", transactions
      .map(lambda x: x[0]).distinct().count())

Unique Bank IDs: 3


In [31]:
#3.Count unique number of customers per Bank ID
customers_per_bank = (
    transactions
    .map(lambda x: (x[0], x[1]))
    .distinct()
    .map(lambda x: (x[0], 1))
    .reduceByKey(lambda a, b: a + b)
)

for bank, count in sorted(customers_per_bank.collect()):
    print(bank, count)

B001 3
B002 2
B003 2


In [32]:
#4.Number of transactions for a given Account Number
account_number = input("Enter Account Number: ").strip()

count = transactions.filter(
    lambda x: x[1] == account_number
).count()

print("Number of transactions:", count)

Enter Account Number:  1001


Number of transactions: 3


In [33]:
#5.Number of credit transactions for a given Account Number in a given year
account_number = input("Enter Account Number: ").strip()
year = input("Enter year (YYYY): ").strip()

credit_count = transactions.filter(
    lambda x: (
        x[1] == account_number
        and x[3].lower() == "credit"
        and x[2].split("-")[2] == year
    )
).count()

print("Number of credit transactions:", credit_count)

Enter Account Number:  1001
Enter year (YYYY):  2025


Number of credit transactions: 1
